# PostgreSQL Logical Backup, Restore, and Verification

[![Open Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Download this notebook, open Colab, and choose **File > Upload notebook**. The full draft course is distributed separately from the public Week 1 repository.


This notebook performs a real PostgreSQL logical backup and restores it into a
different database. The databases are temporary and isolated, so the lab teaches
the full recovery sequence without risking a Supabase project.

**Recovery sequence:** source checks -> dump artifact -> artifact inspection ->
separate restore -> structure checks -> data checks -> behavior check.

No cloud credential is required. The final section translates the same procedure
to Supabase without storing a connection URL.

We use three users, three tickets, and five events: a smaller recovery example,
not the complete Metro Support dataset from earlier weeks. Both databases live on
the same temporary server. That isolates the restore from the source, but does
not protect either database against losing this notebook runtime.

**Your work:** run the example, adapt the supplied known-ticket check, and write
one short recovery account in this notebook. Keep its output, then run cleanup.

## 1. Prepare PostgreSQL in the Notebook Runtime

Google Colab does not start with a PostgreSQL server, so the next cell installs
the free PostgreSQL package when it detects Colab and starts a local service. On a
computer where PostgreSQL is already running, it uses the current local server.

For a local computer, use a disposable PostgreSQL installation and a local Unix
socket (`PGHOST` may name its socket directory). This notebook refuses remote
hosts. Do not point it at a cloud project or a shared production server.

The command prefix is shown explicitly. It changes only because Colab's local
server is owned by its `postgres` operating-system user.

The cells contain **Python** that launches PostgreSQL command-line programs.
`subprocess.run([program, option, value], check=True)` waits for that program and
stops the cell if it fails. `capture_output=True` keeps the result available for
comparison; `text=True` reads it as text. SQL appears inside quoted strings passed
to `psql`. These are three layers, not three versions of SQL.

In [ ]:
import hashlib
import os
from pathlib import Path
import re
import shutil
import subprocess

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if not IN_COLAB and os.environ.get("PGHOST", "") and not os.environ["PGHOST"].startswith("/"):
    raise RuntimeError("Use a local PostgreSQL Unix socket, not a remote host, for this practice notebook.")
if not IN_COLAB and (os.environ.get("PGSERVICE") or os.environ.get("PGHOSTADDR")):
    raise RuntimeError("Unset PGSERVICE and PGHOSTADDR so the local practice target is explicit.")

if IN_COLAB and shutil.which("pg_dump") is None:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "postgresql", "postgresql-client"],
        check=True,
    )

if IN_COLAB:
    subprocess.run(["service", "postgresql", "start"], check=True)

PG_PREFIX = ["sudo", "-u", "postgres"] if IN_COLAB else []
print("Running in Colab:", IN_COLAB)
print("PostgreSQL command prefix:", PG_PREFIX or "current local user")

readiness = subprocess.run(
    PG_PREFIX + ["pg_isready"],
    check=True,
    text=True,
    capture_output=True,
)
print(readiness.stdout.strip())

server_version_result = subprocess.run(
    PG_PREFIX + ["psql", "-X", "--set=ON_ERROR_STOP=on", "--dbname", "postgres",
                 "--tuples-only", "--no-align", "--command", "SHOW server_version_num"],
    check=True,
    text=True,
    capture_output=True,
)
server_version_num = int(server_version_result.stdout.strip())
server_major = server_version_num // 10000

candidate_directories = [
    Path(f"/usr/lib/postgresql/{server_major}/bin"),
    Path(f"/opt/homebrew/opt/postgresql@{server_major}/bin"),
    Path(f"/Applications/Postgres.app/Contents/Versions/{server_major}/bin"),
]
path_pg_dump = shutil.which("pg_dump")
if path_pg_dump:
    candidate_directories.append(Path(path_pg_dump).parent)

PG_BIN = None
for candidate_directory in candidate_directories:
    candidate_dump = candidate_directory / "pg_dump"
    if not candidate_dump.exists():
        continue
    version_text = subprocess.run(
        [str(candidate_dump), "--version"],
        check=True,
        text=True,
        capture_output=True,
    ).stdout
    version_match = re.search(r"(\d+)(?:\.\d+)?", version_text)
    if version_match and int(version_match.group(1)) == server_major:
        PG_BIN = candidate_directory
        break

if PG_BIN is None:
    raise RuntimeError(
        f"For this same-server restore, install PostgreSQL client major version {server_major} "
        "to match the server, then rerun this cell."
    )

PSQL = str(PG_BIN / "psql")
CREATEDB = str(PG_BIN / "createdb")
DROPDB = str(PG_BIN / "dropdb")
PG_DUMP = str(PG_BIN / "pg_dump")
PG_RESTORE = str(PG_BIN / "pg_restore")

print("Server major version:", server_major)
print("Compatible client directory:", PG_BIN)
print(subprocess.run([PG_DUMP, "--version"], check=True, text=True, capture_output=True).stdout.strip())

## 2. Create a Source Database

The source and restore databases have visibly different names. The setup contains
keys, relationships, a status constraint, and a few rows so later checks can test
more than counts.

The SQL is ordinary `CREATE TABLE` and `INSERT`. A unique suffix prevents a new
run from overwriting someone else's database. In Colab, the private artifact
folder is owned by the local `postgres` user so that user can create the dump.
The folder remains private; making it writable by everyone is unnecessary.

In [ ]:
from uuid import uuid4
import tempfile

# Each run owns new database names and a separate artifact directory.
if "SOURCE_DB" in globals():
    raise RuntimeError("Run the final cleanup cell before creating another practice database.")
run_id = uuid4().hex[:8]
SOURCE_DB = "cst4714_recovery_source_" + run_id
RESTORE_DB = "cst4714_recovery_restore_" + run_id
ARTIFACT_DIR = Path(tempfile.mkdtemp(prefix="cst4714-recovery-"))
if IN_COLAB:
    shutil.chown(ARTIFACT_DIR, user="postgres", group="postgres")
SETUP_FILE = ARTIFACT_DIR / "metro_support_setup.sql"
DUMP_FILE = ARTIFACT_DIR / "metro_support.dump"

setup_sql = """
CREATE SCHEMA metro_support;

CREATE TABLE metro_support.users (
    user_id integer PRIMARY KEY,
    display_name text NOT NULL,
    role text NOT NULL
);

CREATE TABLE metro_support.tickets (
    ticket_id integer PRIMARY KEY,
    requester_id integer NOT NULL REFERENCES metro_support.users(user_id),
    status text NOT NULL CONSTRAINT tickets_status_allowed
        CHECK (status IN ('new', 'open', 'in_progress', 'resolved', 'closed')),
    subject text NOT NULL
);

CREATE TABLE metro_support.ticket_events (
    event_id integer PRIMARY KEY,
    ticket_id integer NOT NULL REFERENCES metro_support.tickets(ticket_id),
    event_type text NOT NULL
);

INSERT INTO metro_support.users VALUES
    (101, 'Maya Chen', 'resident'),
    (102, 'Luis Rivera', 'resident'),
    (201, 'Priya Shah', 'agent');

INSERT INTO metro_support.tickets VALUES
    (1001, 101, 'open', 'Streetlight dark near bus stop'),
    (1002, 102, 'in_progress', 'Missed recycling pickup'),
    (1003, 101, 'resolved', 'Low water pressure');

INSERT INTO metro_support.ticket_events VALUES
    (5001, 1001, 'created'),
    (5002, 1001, 'assigned'),
    (5003, 1002, 'created'),
    (5004, 1003, 'created'),
    (5005, 1003, 'status_changed');
"""

SETUP_FILE.write_text(setup_sql, encoding="utf-8")

subprocess.run(PG_PREFIX + [CREATEDB, SOURCE_DB], check=True)
subprocess.run(
    PG_PREFIX
    + [PSQL, "-X", "--set=ON_ERROR_STOP=on", "--dbname", SOURCE_DB, "--file", str(SETUP_FILE)],
    check=True,
    text=True,
    capture_output=True,
)
print("Created source database:", SOURCE_DB)

## 3. Record the Source State

Expected source counts are 3 users, 3 tickets, and 5 events. The three tickets have
different statuses. We record table names, the named status constraint, a check
for missing requesters, and a grouped report as the source baseline. An `assert`
stops the cell if the complete result differs from the expected result.

In [ ]:
source_check_sql = """
SELECT 'table=' || table_name
FROM information_schema.tables
WHERE table_schema = 'metro_support' AND table_type = 'BASE TABLE'
ORDER BY table_name;
SELECT 'users=' || count(*) FROM metro_support.users;
SELECT 'tickets=' || count(*) FROM metro_support.tickets;
SELECT 'ticket_events=' || count(*) FROM metro_support.ticket_events;
SELECT 'constraint=' || conname
FROM pg_constraint
WHERE conname = 'tickets_status_allowed'
  AND conrelid = 'metro_support.tickets'::regclass;
SELECT 'orphan_tickets=' || count(*)
FROM metro_support.tickets AS t
LEFT JOIN metro_support.users AS u ON u.user_id = t.requester_id
WHERE u.user_id IS NULL;
SELECT 'status:' || status || '=' || count(*)
FROM metro_support.tickets GROUP BY status ORDER BY status;
"""

expected_baseline = [
    "table=ticket_events", "table=tickets", "table=users",
    "users=3", "tickets=3", "ticket_events=5",
    "constraint=tickets_status_allowed", "orphan_tickets=0",
    "status:in_progress=1", "status:open=1", "status:resolved=1",
]

source_check = subprocess.run(
    PG_PREFIX + [PSQL, "-X", "--set=ON_ERROR_STOP=on", "--tuples-only", "--no-align",
                 "--dbname", SOURCE_DB, "--command", source_check_sql],
    check=True,
    text=True,
    capture_output=True,
)
print(source_check.stdout.strip())
assert source_check.stdout.strip().splitlines() == expected_baseline

## 4. Create the Logical Backup Artifact

`pg_dump --format=custom` creates an archive for `pg_restore`. The schema option
limits the archive to `metro_support`. This is not a backup of server roles or
other databases. All three related tables come from a consistent source snapshot.

For a custom archive, ownership suppression belongs on **pg_restore**, not
pg_dump. We omit grants at export and omit original ownership at restore so the
classroom example does not depend on identical roles. Restoring application
access would require a separate, reviewed permissions procedure.

In [ ]:
if DUMP_FILE.exists():
    DUMP_FILE.unlink()

subprocess.run(
    PG_PREFIX
    + [
        PG_DUMP,
        "--format=custom",
        "--schema=metro_support",
        "--no-privileges",
        "--file",
        str(DUMP_FILE),
        SOURCE_DB,
    ],
    check=True,
)

dump_bytes = DUMP_FILE.read_bytes()
dump_sha256 = hashlib.sha256(dump_bytes).hexdigest()
print("Dump path:", DUMP_FILE)
print("Dump size in bytes:", len(dump_bytes))
print("SHA-256:", dump_sha256)
assert len(dump_bytes) > 0

## 5. Inspect the Artifact Before Restoring

`pg_restore --list` reads the archive table of contents. Seeing the expected
schema, tables, data, constraints, and indexes confirms the archive inventory;
only a separate restore tests whether PostgreSQL can use it.

A checksum can detect whether a file changed when compared with a trusted earlier
checksum. It does not prove the source was correct, complete, or safe. A dump can
contain executable SQL; use only the archive created by this notebook here.

In [ ]:
archive_list = subprocess.run(
    PG_PREFIX + [PG_RESTORE, "--list", str(DUMP_FILE)],
    check=True,
    text=True,
    capture_output=True,
)

important_lines = [
    line
    for line in archive_list.stdout.splitlines()
    if any(term in line for term in ("SCHEMA", "TABLE ", "TABLE DATA", "CONSTRAINT", "INDEX"))
]
print("\n".join(important_lines))

## 6. Restore Into a Different Database

The destination is empty and separate. `--exit-on-error` prevents an archive with
an early failure from looking successful merely because later items continued.
For this small restore, `--single-transaction` also prevents a failed operation
from leaving a partly restored set of transactional objects. It does not remove
preexisting target objects; start with the empty database created below.

In [ ]:
subprocess.run(PG_PREFIX + [CREATEDB, RESTORE_DB], check=True)

restore_result = subprocess.run(
    PG_PREFIX
    + [
        PG_RESTORE,
        "--exit-on-error",
        "--single-transaction",
        "--no-owner",
        "--no-privileges",
        "--dbname",
        RESTORE_DB,
        str(DUMP_FILE),
    ],
    check=True,
    text=True,
    capture_output=True,
)

print("Restored into separate database:", RESTORE_DB)
print("Restore exit code:", restore_result.returncode)

## 7. Verify Structure, Data, and Relationships

The following checks ask different questions:

- Do all three tables exist?
- Do row counts match the source baseline?
- Are there tickets with a missing requester relationship?
- Does a meaningful report return the expected grouped result?

In [ ]:
# Run exactly the same SQL against a different database.
restore_check = subprocess.run(
    PG_PREFIX + [PSQL, "-X", "--set=ON_ERROR_STOP=on", "--tuples-only", "--no-align",
                 "--dbname", RESTORE_DB, "--command", source_check_sql],
    check=True,
    text=True,
    capture_output=True,
)
print(restore_check.stdout.strip())
assert restore_check.stdout.strip().splitlines() == expected_baseline
assert restore_check.stdout == source_check.stdout
print("The complete baseline and restored results agree.")

## 8. Verify Behavior With an Expected Failure

A restored table can contain rows while missing an integrity rule. This insert
must fail because `almost_done` is not an allowed status. A disconnected server
or a misspelled table would also produce an error, but neither proves the rule
works. We require PostgreSQL error code **23514** (check violation) and the exact
constraint name. The attempted write is inside a transaction that cannot commit.

In [ ]:
invalid_insert = subprocess.run(
    PG_PREFIX
    + [
        PSQL,
        "-X",
        "--set=ON_ERROR_STOP=on",
        "--set=VERBOSITY=verbose",
        "--dbname",
        RESTORE_DB,
        "--command",
        """
        BEGIN;
        INSERT INTO metro_support.tickets
            (ticket_id, requester_id, status, subject)
        VALUES
            (1099, 101, 'almost_done', 'Constraint restore test');
        ROLLBACK;
        """,
    ],
    check=False,
    text=True,
    capture_output=True,
)

print("Constraint-test exit code:", invalid_insert.returncode)
print(invalid_insert.stderr.strip())
assert invalid_insert.returncode != 0, "The invalid status was not rejected."
assert "23514" in invalid_insert.stderr, "This was a different SQL failure, not a check violation."
assert '"tickets_status_allowed"' in invalid_insert.stderr

after_test = subprocess.run(
    PG_PREFIX + [PSQL, "-X", "--set=ON_ERROR_STOP=on", "--tuples-only", "--no-align",
                 "--dbname", RESTORE_DB, "--command",
                 "SELECT count(*) FROM metro_support.tickets WHERE ticket_id = 1099;"],
    check=True, text=True, capture_output=True,
)
assert after_test.stdout.strip() == "0"
print("The named status rule rejected the write; test ticket 1099 was not retained.")

## Your Check: Compare One Known Ticket

Counts would stay the same if a ticket's subject or requester changed. The query
below therefore compares one ticket's values and its requester's name. The loop
runs the same query once against the source and once against the restore.

First run the example for ticket 1001. Then change `CHECK_TICKET_ID` to **1002 or
1003** and run it again. Read that ticket's original `INSERT` above and check that
the result matches the intended record, not just the other database. Explain what
your check would catch that the counts would miss. You can extend the selected
columns if you want to test a different meaningful property; no new report is needed.

In [ ]:
CHECK_TICKET_ID = 1001  # Change to 1002 or 1003 for your check.
assert CHECK_TICKET_ID in (1001, 1002, 1003)
known_ticket_sql = f"""
SELECT t.ticket_id, t.status, t.subject, u.display_name AS requester
FROM metro_support.tickets AS t
JOIN metro_support.users AS u ON u.user_id = t.requester_id
WHERE t.ticket_id = {CHECK_TICKET_ID};
"""

known_results = []
for database_name in (SOURCE_DB, RESTORE_DB):
    result = subprocess.run(
        PG_PREFIX + [PSQL, "-X", "--set=ON_ERROR_STOP=on", "--tuples-only", "--no-align",
                     "--dbname", database_name, "--command", known_ticket_sql],
        check=True, text=True, capture_output=True,
    )
    print(database_name, "->", result.stdout.strip())
    known_results.append(result.stdout.strip())

assert known_results[0] and known_results[0] == known_results[1]
print("This ticket's selected values match in source and restore.")

## Connection Reference: What Changes for Supabase?

Supabase Free projects do not receive the automatic database backups described
for paid plans. A course recovery plan therefore uses a logical connection and
runtime credential. The database tools stay the same; the host, port, user,
database, and TLS configuration change. This reference is not another required
cloud task. Never use the remote source as the restore target.

Obtain non-password `PGHOST`, `PGPORT`, `PGUSER`, and `PGDATABASE` values from
the approved connection panel. `--password` prompts rather than placing the
password in a visible connection URL:

```bash
pg_dump --format=custom --schema=metro_support --no-privileges \
  --host="$PGHOST" --port="$PGPORT" --username="$PGUSER" \
  --dbname="$PGDATABASE" --password --file=metro_support.dump
```

Use the current Supabase connection guidance. A direct endpoint may require IPv6;
the **session pooler** offers an IPv4-compatible path. Use the documented TLS
configuration; do not disable certificate verification to fix an unreachable
network. Use a matching-major client. Chapter 8 supplies the separate
`RESTORE_PGHOST`, `RESTORE_PGPORT`, `RESTORE_PGUSER`, and `RESTORE_PGDATABASE`
restore command with `--single-transaction`, `--no-owner`, and `--no-privileges`.
This schema archive does not include all Supabase Auth, Storage, or project settings.

Official free resources:

- <https://supabase.com/docs/guides/platform/backups>
- <https://supabase.com/docs/guides/database/connecting-to-postgres>
- <https://www.postgresql.org/docs/current/backup-dump.html>

## Recovery Record: Complete Before Submission

Write a short recovery account in this cell. Name the separate restore target,
summarize the supplied checks, explain your additional known-record check, and
identify one thing still untested. State which connection would change if
Supabase were the source. Use the results already above; do not copy them into
another table or create a second report.

**License:** prose CC BY-NC-SA 4.0; code MIT; synthetic data CC0.

## Finish: Remove Only This Run's Practice Resources

After completing your check and recovery account, keep the output in the notebook
and run this cell. It removes the two uniquely named practice databases and their
temporary artifact folder. The dump is not a submission or a retained recovery
copy. In Colab it also stops the service started for this practice; on your own
computer it leaves the preexisting PostgreSQL service running.

If an earlier cell fails, fix the stated problem or use this cleanup before
starting a fresh run. Do not substitute the name of a database you already use.

In [ ]:
for database_name in (RESTORE_DB, SOURCE_DB):
    assert database_name in (
        "cst4714_recovery_source_" + run_id,
        "cst4714_recovery_restore_" + run_id,
    )
    subprocess.run(
        PG_PREFIX + [DROPDB, "--if-exists", database_name],
        check=True, text=True, capture_output=True,
    )

shutil.rmtree(ARTIFACT_DIR)
if IN_COLAB:
    subprocess.run(["service", "postgresql", "stop"], check=True)
del SOURCE_DB, RESTORE_DB
print("Removed this run's practice databases and artifact folder.")
print("Colab practice service stopped." if IN_COLAB else "Existing local service left running.")